In [ ]:
# Conditioning on the particle type, masking the velocity field

In [75]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm
from torchdyn.core import NeuralODE
from torchcfm.conditional_flow_matching import ExactOptimalTransportConditionalFlowMatcher
from matplotlib import animation
import numpy as np
import load_and_preprocess as lap
import train_and_eval_functions as taef
import os
import matplotlib.pyplot as plt
import math
from sklearn.metrics import roc_curve, auc
from sklearn.model_selection import train_test_split
import h5py

In [76]:
label = 'otfm_EB_sigma0p01_v2'
save_path = '/global/homes/m/mcohen54/ctfm_development/trained_models/trial_5'

batchSize = 512
numberOfEpochs = 20
patience = 5

pxpypz = True
p_train = 0.5
p_test = 0.25
plots_path = save_path+'/plots'

input_dim = 3
model_dim = 128
ff_dim = 128
num_heads = 4
num_layers = 4
n_mask_vals = 5 # 4 particle types + 1 padding

flowSigma = 0.01

trained = True
multiGPU = True


if not multiGPU:
    os.environ["CUDA_VISIBLE_DEVICES"]="0"

In [77]:
def load_subdicts_from_h5(save_dir, tags_to_use=None):
    """
    Loads sub-dictionaries of NumPy arrays from HDF5 files in a directory and reconstructs the original structure.
    
    Args:
        save_dir (str): The directory where the HDF5 files are stored.
    
    Returns:
        main_dict (dict): A dictionary of dictionaries where the innermost values are NumPy arrays.
    """
    main_dict = {}
    
    for filename in os.listdir(save_dir):
        if filename.endswith(".h5") and not filename.startswith('.'):

            
            sub_dict_name = os.path.splitext(filename)[0]
            if tags_to_use is not None and sub_dict_name not in tags_to_use:
                continue
            file_path = os.path.join(save_dir, filename)
            with h5py.File(file_path, 'r') as f:
                sub_dict = {key: np.array(f[key]) for key in f}
            main_dict[sub_dict_name] = sub_dict
            print(f"Loaded {sub_dict_name} from {file_path}")
    
    return main_dict

In [78]:
def combine_data(datasets, tags_to_combine, new_tag, delete_old_tags=True):
    """
    Combines subdicts of the 'datasets' dict.
    
    Inputs: 
        datasets: dict that maps {dataset_tag : dataset_dict}
        tags_to_combine: list if strings [dataset_tag1, ..., dataset_tagN] of the tags to be combined
        new_tag: the name of the new tag of the combined subdict

    Returns: 
        datasets: same datasets dict as input, but with the specified tags combined.
    """

    # initialize empty lists for new tag
    datasets[new_tag] = {key: [] for key in datasets[tags_to_combine[0]].keys()}

    # Loop through old tags and append np arrays to lists
    for tag in tags_to_combine:
        for key, value in datasets[tag].items():
            datasets[new_tag][key].append(value)

    # Concatenate lists into single np array
    for key, value in datasets[new_tag].items():
        datasets[new_tag][key] = np.concatenate(value, axis=0)

    # Delete old tags
    if delete_old_tags:
        for tag in tags_to_combine:
            del datasets[tag]

    # Make sure everything is an np array
    for tag, data_dict in datasets.items():
        for key, value in data_dict.items():
            data_dict[key] = np.array(value)

    return datasets

In [79]:
label_map = {
    'mc23e_HNLeemu' : r"$\mathrm{HNL}\;\to\; e\,e\,\mu,\;m=7.5\,\mathrm{GeV},\;\tau=1\,\mathrm{ns}$",
    'mc23e_HAHMggf' : r"$\mathrm{HAHM}\,ggF\,h\to Z_{d}Z_{d}\to2\ell2\nu, m_{h}=125\,\mathrm{GeV}, m_{Z_{d}}=28\,\mathrm{GeV}$",
    #'mc23e_new_HtoSUEP_VBF_fullhad_125_3p00_4p00_VBFFilter' : r'VBF $h\to\mathrm{SUEP}\to\text{full-had}$ (VBF filter)',
    'mc23e_new_HtoSUEP_ggH_fullhad_125_3p00_3p00_noFilter' : r'ggF $h\to\mathrm{SUEP}\to\text{full-had}$',
    'mc23e_new_VBF_H125_a55a55_4b_ctau1_filtered' : r'VBF $h\to a_{55~\mathrm{GeV}}a_{55~\mathrm{GeV}} \to 4b$, $\tau_a = 1$ ns',
    'mc23e_new_Znunu_FxFx3jHT2bias_SW_pTvv70_BFilter' : r'$Z\to\nu\nu$ ($b$ filter)',
    'mc23e_new_ggF_H125_a16a16_4b_ctau10_filtered' : r'ggF $h\to a_{16~\mathrm{GeV}}a_{16~\mathrm{GeV}} \to 4b$, $\tau_a = 10$ ns (jet-$p_{\mathrm{T}}$ filter)',
    'mc23e_new_hh_bbbb_vbf_novhh_5fs_l1cvv1cv1' : r'VBF $hh\to 4b$',
    'EB': 'EB'
}
datasets = load_subdicts_from_h5(save_dir='/pscratch/sd/m/mcohen54/data/loaded_and_matched_data/topo2A_datasets')
tags_to_remove = [key for key in datasets if key not in label_map]
for tag in tags_to_remove:
    del datasets[tag]

Loaded mc23e_HAHMggf from /pscratch/sd/m/mcohen54/data/loaded_and_matched_data/topo2A_datasets/mc23e_HAHMggf.h5
Loaded mc23e_new_CC_directHlR23_150 from /pscratch/sd/m/mcohen54/data/loaded_and_matched_data/topo2A_datasets/mc23e_new_CC_directHlR23_150.h5
Loaded mc23e_new_HtoSUEP_ggH_fullhad_125_3p00_3p00_noFilter from /pscratch/sd/m/mcohen54/data/loaded_and_matched_data/topo2A_datasets/mc23e_new_HtoSUEP_ggH_fullhad_125_3p00_3p00_noFilter.h5
Loaded mc23e_new_HAHM_S2Zd4e_60_25_0p1ns from /pscratch/sd/m/mcohen54/data/loaded_and_matched_data/topo2A_datasets/mc23e_new_HAHM_S2Zd4e_60_25_0p1ns.h5
Loaded mc23e_new_hh_bbbb_vbf_novhh_5fs_l1cvv2cv1 from /pscratch/sd/m/mcohen54/data/loaded_and_matched_data/topo2A_datasets/mc23e_new_hh_bbbb_vbf_novhh_5fs_l1cvv2cv1.h5
Loaded mc23e_ChiPlusChiMinus100_99_0p3ns from /pscratch/sd/m/mcohen54/data/loaded_and_matched_data/topo2A_datasets/mc23e_ChiPlusChiMinus100_99_0p3ns.h5
Loaded mc23e_ChiPlusChiMinus500_40_10ns from /pscratch/sd/m/mcohen54/data/loaded_and

In [80]:
label_map = {
    'mc23e_HNLeemu' : r"$\mathrm{HNL}\;\to\; e\,e\,\mu,\;m=7.5\,\mathrm{GeV},\;\tau=1\,\mathrm{ns}$",
    'mc23e_HAHMggf' : r"$\mathrm{HAHM}\,ggF\,h\to Z_{d}Z_{d}\to2\ell2\nu, m_{h}=125\,\mathrm{GeV}, m_{Z_{d}}=28\,\mathrm{GeV}$",
    #'mc23e_new_HtoSUEP_VBF_fullhad_125_3p00_4p00_VBFFilter' : r'VBF $h\to\mathrm{SUEP}\to\text{full-had}$ (VBF filter)',
    'mc23e_new_HtoSUEP_ggH_fullhad_125_3p00_3p00_noFilter' : r'ggF $h\to\mathrm{SUEP}\to\text{full-had}$',
    'mc23e_new_VBF_H125_a55a55_4b_ctau1_filtered' : r'VBF $h\to a_{55~\mathrm{GeV}}a_{55~\mathrm{GeV}} \to 4b$, $\tau_a = 1$ ns',
    'mc23e_new_Znunu_FxFx3jHT2bias_SW_pTvv70_BFilter' : r'$Z\to\nu\nu$ ($b$ filter)',
    'mc23e_new_ggF_H125_a16a16_4b_ctau10_filtered' : r'ggF $h\to a_{16~\mathrm{GeV}}a_{16~\mathrm{GeV}} \to 4b$, $\tau_a = 10$ ns (jet-$p_{\mathrm{T}}$ filter)',
    'mc23e_new_hh_bbbb_vbf_novhh_5fs_l1cvv1cv1' : r'VBF $hh\to 4b$',
    'EB': 'EB',
    'EB_473255': 1,
    'EB_475321': 1,
    'EB_482596': 1,
    'topo2A_train': 1,
}
HLT_datasets = load_subdicts_from_h5(save_dir='/pscratch/sd/m/mcohen54/data/loaded_and_matched_data/datasets')
tags_to_remove = [key for key in HLT_datasets if key not in label_map]
for tag in tags_to_remove:
    del HLT_datasets[tag]

print(HLT_datasets.keys())

HLT_datasets = combine_data(HLT_datasets, ['EB_473255', 'EB_475321', 'EB_482596', 'topo2A_train'], 'EB')

Loaded mc23e_HAHMggf from /pscratch/sd/m/mcohen54/data/loaded_and_matched_data/datasets/mc23e_HAHMggf.h5
Loaded topo2A_train from /pscratch/sd/m/mcohen54/data/loaded_and_matched_data/datasets/topo2A_train.h5
Loaded mc23e_new_CC_directHlR23_150 from /pscratch/sd/m/mcohen54/data/loaded_and_matched_data/datasets/mc23e_new_CC_directHlR23_150.h5
Loaded mc23e_new_HtoSUEP_ggH_fullhad_125_3p00_3p00_noFilter from /pscratch/sd/m/mcohen54/data/loaded_and_matched_data/datasets/mc23e_new_HtoSUEP_ggH_fullhad_125_3p00_3p00_noFilter.h5
Loaded mc23e_new_HAHM_S2Zd4e_60_25_0p1ns from /pscratch/sd/m/mcohen54/data/loaded_and_matched_data/datasets/mc23e_new_HAHM_S2Zd4e_60_25_0p1ns.h5
Loaded mc23e_new_hh_bbbb_vbf_novhh_5fs_l1cvv2cv1 from /pscratch/sd/m/mcohen54/data/loaded_and_matched_data/datasets/mc23e_new_hh_bbbb_vbf_novhh_5fs_l1cvv2cv1.h5
Loaded EB_475321 from /pscratch/sd/m/mcohen54/data/loaded_and_matched_data/datasets/EB_475321.h5
Loaded mc23e_ChiPlusChiMinus100_99_0p3ns from /pscratch/sd/m/mcohen54/d

In [81]:
print(datasets.keys())

dict_keys(['mc23e_HAHMggf', 'mc23e_new_HtoSUEP_ggH_fullhad_125_3p00_3p00_noFilter', 'EB', 'mc23e_HNLeemu', 'mc23e_new_VBF_H125_a55a55_4b_ctau1_filtered', 'mc23e_new_hh_bbbb_vbf_novhh_5fs_l1cvv1cv1', 'mc23e_new_ggF_H125_a16a16_4b_ctau10_filtered', 'mc23e_new_Znunu_FxFx3jHT2bias_SW_pTvv70_BFilter'])


In [82]:
for key, data_dict in datasets.items():
    if key not in HLT_datasets:
        print(f"Tag {key} not in HLT_datasets, skipping.")
        continue
    
    print(key)
    HLT_data_dict = HLT_datasets[key]
    print(f"{key} event numbers: {data_dict['event_numbers'][:5]} and {HLT_data_dict['event_numbers'][-5:]}")
    print(f"{key} run numbers: {data_dict['run_numbers'][:5]} and {HLT_data_dict['run_numbers'][-5:]}")


    print(f"HLT event numbers: {HLT_data_dict['event_numbers'][:5]} and {HLT_data_dict['event_numbers'][-5:]}")
    print(f"HLT run numbers: {HLT_data_dict['run_numbers'][:5]} and {HLT_data_dict['run_numbers'][-5:]}")
    print()

mc23e_HAHMggf
mc23e_HAHMggf event numbers: [20001 20010 20012 20008 20009] and [ 99995  99999  99992 100000  99996]
mc23e_HAHMggf run numbers: [470000 470000 470000 470000 470000] and [470000 470000 470000 470000 470000]
HLT event numbers: [20001 20010 20012 20008 20009] and [ 99995  99999  99992 100000  99996]
HLT run numbers: [470000 470000 470000 470000 470000] and [470000 470000 470000 470000 470000]

mc23e_new_HtoSUEP_ggH_fullhad_125_3p00_3p00_noFilter
mc23e_new_HtoSUEP_ggH_fullhad_125_3p00_3p00_noFilter event numbers: [4001 4010 4020 4002 4004] and [49998 49996 49999 49991 49997]
mc23e_new_HtoSUEP_ggH_fullhad_125_3p00_3p00_noFilter run numbers: [470000 470000 470000 470000 470000] and [470000 470000 470000 470000 470000]
HLT event numbers: [4001 4010 4020 4002 4004] and [49998 49996 49999 49991 49997]
HLT run numbers: [470000 470000 470000 470000 470000] and [470000 470000 470000 470000 470000]

EB
EB event numbers: [448065571 448036459 448023118 448030614 448015275] and [4617829

In [86]:

for key in datasets.keys():
    # Pre‐build a dict of HLT weights keyed by (event, run) tuples
    hlt_dict = {
        (int(e), int(r)): w
        for e, r, w in zip(
            HLT_datasets[key]['event_numbers'],
            HLT_datasets[key]['run_numbers'],
            HLT_datasets[key]['weights']
        )
    }
    
    
    ev_d   = datasets[key]['event_numbers']
    run_d  = datasets[key]['run_numbers']
    
    # Look up weight for each (event, run), defaulting to 0.0 if not found
    matched_weights = np.array([
        hlt_dict.get((int(e), int(r)), 0.0)
        for e, r in zip(ev_d, run_d)
    ])
    
    datasets[key]['weights'] = matched_weights
    print(f"Assigned weights for {key}: "
          f"{np.count_nonzero(matched_weights)} matched out of {len(matched_weights)}")
    

Assigned weights for mc23e_HAHMggf: 100000 matched out of 100000
Assigned weights for mc23e_new_HtoSUEP_ggH_fullhad_125_3p00_3p00_noFilter: 100000 matched out of 100000
Assigned weights for EB: 2536972 matched out of 3634169
Assigned weights for mc23e_HNLeemu: 100000 matched out of 100000
Assigned weights for mc23e_new_VBF_H125_a55a55_4b_ctau1_filtered: 100000 matched out of 100000
Assigned weights for mc23e_new_hh_bbbb_vbf_novhh_5fs_l1cvv1cv1: 50000 matched out of 50000
Assigned weights for mc23e_new_ggF_H125_a16a16_4b_ctau10_filtered: 10000 matched out of 10000
Assigned weights for mc23e_new_Znunu_FxFx3jHT2bias_SW_pTvv70_BFilter: 10000 matched out of 10000


In [87]:
for tag in datasets.keys():
    print(tag)
    nonzero_weight_mask = datasets[tag]['weights'] != 0
    datasets[tag] = {key: value[nonzero_weight_mask] for key, value in datasets[tag].items()}

mc23e_HAHMggf
mc23e_new_HtoSUEP_ggH_fullhad_125_3p00_3p00_noFilter
EB
mc23e_HNLeemu
mc23e_new_VBF_H125_a55a55_4b_ctau1_filtered
mc23e_new_hh_bbbb_vbf_novhh_5fs_l1cvv1cv1
mc23e_new_ggF_H125_a16a16_4b_ctau10_filtered
mc23e_new_Znunu_FxFx3jHT2bias_SW_pTvv70_BFilter


In [91]:
idxs = [2141, 234151, 657436]
run_nums = datasets['EB']['run_numbers'][idxs]
ev_nums = datasets['EB']['event_numbers'][idxs]
weights = datasets['EB']['weights'][idxs]

for r, e, w in zip(run_nums, ev_nums, weights):
    hlt_rs = HLT_datasets['EB']['run_numbers']
    hlt_es = HLT_datasets['EB']['event_numbers']
    hlt_ws = HLT_datasets['EB']['weights']
    mask = np.where((hlt_rs == r) & (hlt_es == e))
    print(hlt_ws[mask])

print(weights)

[18360.8]
[152520.]
[152520.]
[ 18360.8 152520.  152520. ]


In [92]:
for tag, data_dict in datasets.items():
    print(tag)
    for key, value in data_dict.items():
        print(f' {key}: {value.shape}')
    

mc23e_HAHMggf
 data: (100000, 44)
 event_numbers: (100000,)
 run_numbers: (100000,)
 topo2A_AD_scores: (100000,)
 weights: (100000,)
mc23e_new_HtoSUEP_ggH_fullhad_125_3p00_3p00_noFilter
 data: (100000, 44)
 event_numbers: (100000,)
 run_numbers: (100000,)
 topo2A_AD_scores: (100000,)
 weights: (100000,)
EB
 data: (2536972, 44)
 event_numbers: (2536972,)
 run_numbers: (2536972,)
 topo2A_AD_scores: (2536972,)
 weights: (2536972,)
mc23e_HNLeemu
 data: (100000, 44)
 event_numbers: (100000,)
 run_numbers: (100000,)
 topo2A_AD_scores: (100000,)
 weights: (100000,)
mc23e_new_VBF_H125_a55a55_4b_ctau1_filtered
 data: (100000, 44)
 event_numbers: (100000,)
 run_numbers: (100000,)
 topo2A_AD_scores: (100000,)
 weights: (100000,)
mc23e_new_hh_bbbb_vbf_novhh_5fs_l1cvv1cv1
 data: (50000, 44)
 event_numbers: (50000,)
 run_numbers: (50000,)
 topo2A_AD_scores: (50000,)
 weights: (50000,)
mc23e_new_ggF_H125_a16a16_4b_ctau10_filtered
 data: (10000, 44)
 event_numbers: (10000,)
 run_numbers: (10000,)
 top

In [93]:
def save_subdicts_to_h5(main_dict, save_dir):
    """
    Saves each sub-dictionary of NumPy arrays in the main_dict to separate HDF5 files.

    Args:
        main_dict (dict): A dictionary of dictionaries where the innermost values are NumPy arrays.
        save_dir (str): The directory where the HDF5 files will be saved.
    """
    # Ensure the save directory exists
    os.makedirs(save_dir, exist_ok=True)

    for sub_dict_name, sub_dict in main_dict.items():
        file_path = os.path.join(save_dir, f"{sub_dict_name}.h5")
        with h5py.File(file_path, 'w') as f:
            for key, arr in sub_dict.items():
                f.create_dataset(key, data=arr)
        print(f"Saved {sub_dict_name} to {file_path}")

In [94]:
save_subdicts_to_h5(datasets, save_dir='/pscratch/sd/m/mcohen54/data/loaded_and_matched_data/matched_datasets')

Saved mc23e_HAHMggf to /pscratch/sd/m/mcohen54/data/loaded_and_matched_data/matched_datasets/mc23e_HAHMggf.h5
Saved mc23e_new_HtoSUEP_ggH_fullhad_125_3p00_3p00_noFilter to /pscratch/sd/m/mcohen54/data/loaded_and_matched_data/matched_datasets/mc23e_new_HtoSUEP_ggH_fullhad_125_3p00_3p00_noFilter.h5
Saved EB to /pscratch/sd/m/mcohen54/data/loaded_and_matched_data/matched_datasets/EB.h5
Saved mc23e_HNLeemu to /pscratch/sd/m/mcohen54/data/loaded_and_matched_data/matched_datasets/mc23e_HNLeemu.h5
Saved mc23e_new_VBF_H125_a55a55_4b_ctau1_filtered to /pscratch/sd/m/mcohen54/data/loaded_and_matched_data/matched_datasets/mc23e_new_VBF_H125_a55a55_4b_ctau1_filtered.h5
Saved mc23e_new_hh_bbbb_vbf_novhh_5fs_l1cvv1cv1 to /pscratch/sd/m/mcohen54/data/loaded_and_matched_data/matched_datasets/mc23e_new_hh_bbbb_vbf_novhh_5fs_l1cvv1cv1.h5
Saved mc23e_new_ggF_H125_a16a16_4b_ctau10_filtered to /pscratch/sd/m/mcohen54/data/loaded_and_matched_data/matched_datasets/mc23e_new_ggF_H125_a16a16_4b_ctau10_filtered